# Using ALSO in practice

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Optional, Tuple, Callable
from tqdm.auto import tqdm

In [2]:
import torch
import numpy as np


def set_global_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
set_global_seed(42)

## Data Preparation

We will use Kaggle Credit Card Fraud data set from [TensorFlow tutorial](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data).

In [4]:
raw_df = pd.read_csv('https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv')
raw_df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


First we will process dataset in a standard way

In [5]:
cleaned_df = raw_df.copy()
cleaned_df.pop('Time')
eps = 0.001
cleaned_df['Log Amount'] = np.log(cleaned_df.pop('Amount') + eps)

And make train test split

In [6]:
train_df, test_df = train_test_split(cleaned_df, test_size=0.2)
train_labels = np.array(train_df.pop('Class')).reshape(-1, 1)
bool_train_labels = train_labels[:, 0] != 0
test_labels = np.array(test_df.pop('Class')).reshape(-1, 1)

train_features = np.array(train_df)
test_features = np.array(test_df)

Then we will normalize features

In [7]:
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

train_features = np.clip(train_features, -5, 5)
test_features = np.clip(test_features, -5, 5)

## Training With Object-level Weighting

In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [9]:
train_dataset = TensorDataset(torch.from_numpy(train_features).to(torch.float32), torch.from_numpy(train_labels).to(torch.long))
test_dataset = TensorDataset(torch.from_numpy(test_features).to(torch.float32), torch.from_numpy(test_labels).to(torch.long))

Now we will train some model with sample-level weighting. To make it compatible with ALSO interface, we need to return the index of the sample with it. To do it, we will use new wrapper dataset.

In [10]:
# Define a custom Dataset wrapper that returns the sample index along with the data.
# This is necessary for ALSO's object-level weighting, as the optimizer needs to know
# exactly which sample in the full dataset it is processing.

class IndexedDataset(torch.utils.data.Dataset):

    def __init__(
        self, dataset: torch.utils.data.Dataset, transform: Optional[Callable] = None
    ):
        self._dataset = dataset
        self.transform = transform

    def __len__(self) -> int:
        return len(self._dataset)

    def __getitem__(self, i: int) -> Tuple[Tuple[torch.Tensor, torch.Tensor], int]:
        X, y = self._dataset[i]
        if self.transform is not None:
            X = self.transform(X)
        return (X, y), i

In [11]:
train_dataset = IndexedDataset(train_dataset)
test_dataset = IndexedDataset(test_dataset)

As a model we will use 2-layer MLP

In [12]:
from torch import nn

In [13]:
set_global_seed(42)
model = nn.Sequential(
    nn.Linear(train_dataset[0][0][0].shape[0], 128),
    nn.ReLU(),
    nn.Linear(128, 1),
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=512)
test_loader = DataLoader(test_dataset, batch_size=512)

Let's import ALSO

In [15]:
from also import ALSO

In [16]:
optimizer = ALSO(
    params=model.parameters(),      # The model parameters to optimize.
    n_groups=len(train_dataset),    # For object-level weighting, each sample is its own group.
    batch_size=512,                 # The batch size used in the DataLoader.
)

In [17]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def compute_roc_auc(model, dataloder, device):
    predictions = []
    targets = []
    for (X, y), _ in tqdm(dataloder):
        X, y = X.to(device), y.to(device)
        preds = model(X)
        predictions.extend(preds.flatten().cpu().tolist())
        targets.extend(y.flatten().cpu().tolist())
    return roc_auc_score(targets, predictions)

Now we are ready to write a train-loop using ALSO. ALSO's step utilize a specific [closure function](https://docs.pytorch.org/docs/stable/optim.html#optimizer-step-closure). Additionally, ALSO needs per-sample losses. Thus there are two modification of the train-loop to make it compatible with ALSO:

1) We need to wrap gradient computation into closure form, which returns per-sample losses and loss to log
2) We need to use loss with `reduction='none'`

In [18]:
# Define the loss function. We use BCEWithLogitsLoss which is numerically stable.
# IMPORTANT: `reduction='none'` is required for ALSO, as it needs the loss value for each individual sample.
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [19]:
n_epoches = 1
model.to(device)
for epoch in range(n_epoches):
    for (X, y), indexes in tqdm(train_loader):
        X, y = X.to(device), y.to(device)
        def closure(w, scale):
            # Define the closure function required by ALSO.step().
            # This function encapsulates the forward pass, loss computation, and backward pass.
            optimizer.zero_grad() # Clear previous gradients.
            preds = model(X).flatten() # Get model predictions.
            losses = loss_fn(preds, y.flatten().float()) # Calculate per-sample losses.
            losses = losses * scale # Apply scaling factor for unbiased gradients.
            loss = (w * losses).sum() # Apply the weights (w) provided by ALSO and sum them up.
            loss.backward() # Compute gradients.
            loss_log = losses.mean().item() # Calculate a loss value for logging.
            return losses, loss_log
        
        # Perform an optimization step.
        # We pass the closure and the `groups_indexes` (which are the sample indices for object-level weighting).
        optimizer.step(closure=closure, groups_indexes=indexes.to(device))

  0%|          | 0/446 [00:00<?, ?it/s]

In [20]:
print("Test roc-auc: ", compute_roc_auc(model, test_loader, device))

  0%|          | 0/112 [00:00<?, ?it/s]

Test roc-auc:  0.9027267350384162


## Training With Class-level Weighting

To train model with class-level weighting we need to slightly modify our pipeline: use `n_groups` parameter in ALSO equal to number of classes and use `groups_indexes` parameter equal to object classes. Let's modify our code

In [21]:
set_global_seed(42)
model = nn.Sequential(
    nn.Linear(train_dataset[0][0][0].shape[0], 128),
    nn.ReLU(),
    nn.Linear(128, 1),
)

In [22]:
optimizer = ALSO(
    params=model.parameters(),
    n_groups=2, # The number of groups is now 2 (class 0 and class 1).
    batch_size=512,
)

In [23]:
n_epoches = 1
model.to(device)
for epoch in range(n_epoches):
    for (X, y), _ in tqdm(train_loader): # We don't need the sample 'indexes' here.
        X, y = X.to(device), y.to(device)
        def closure(w, scale):
            optimizer.zero_grad()
            preds = model(X).flatten()
            losses = loss_fn(preds, y.flatten().float())
            losses = losses * scale
            loss = (w * losses).sum()
            loss.backward()
            loss_log = losses.mean().item()
            return losses, loss_log
        # The key difference is here: we pass the class labels `y` as the `groups_indexes`.
        # This tells ALSO to group samples by their class and apply class-level weights.
        optimizer.step(closure=closure, groups_indexes=y)

  0%|          | 0/446 [00:00<?, ?it/s]

In [24]:
print("Test roc-auc: ", compute_roc_auc(model, test_loader, device))

  0%|          | 0/112 [00:00<?, ?it/s]

Test roc-auc:  0.9019450633376592


## Training With Class-level Weighting and Static Weights Init

Now let's add initialization from inverse proportion of the class to ALSO

In [25]:
# Calculate initial weights for each class based on inverse class frequency.
# This is a common heuristic for handling class imbalance.
neg_count = (train_labels.flatten() == 0).sum()
pos_count = (train_labels.flatten() == 1).sum()
total_count = len(train_labels)
weights_raw = (total_count / neg_count, total_count / pos_count)
weights = torch.tensor(weights_raw, dtype=torch.float32, device=device)
weights = weights / weights.sum()

In [26]:
set_global_seed(42)
model = nn.Sequential(
    nn.Linear(train_dataset[0][0][0].shape[0], 128),
    nn.ReLU(),
    nn.Linear(128, 1),
)

In [27]:
optimizer = ALSO(
    params=model.parameters(),
    n_groups=2, # as we use class-level weighting
    batch_size=512,
    pi_reg=weights, # Set the initial weights for the groups.
    pi_init=weights # Use the same weights for regularization, encouraging learned weights to stay near these values.
)

In [28]:
n_epoches = 1
model.to(device)
for epoch in range(n_epoches):
    for (X, y), _ in tqdm(train_loader):
        X, y = X.to(device), y.to(device)
        def closure(w, scale):
            optimizer.zero_grad()
            preds = model(X).flatten()
            losses = loss_fn(preds, y.flatten().float())
            losses = losses * scale
            loss = (w * losses).sum()
            loss.backward()
            loss_log = losses.mean().item()
            return losses, loss_log
        optimizer.step(closure=closure, groups_indexes=y)

  0%|          | 0/446 [00:00<?, ?it/s]

In [29]:
print("Test roc-auc: ", compute_roc_auc(model, test_loader, device))

  0%|          | 0/112 [00:00<?, ?it/s]

Test roc-auc:  0.928167582804084


## Conclusion

Now you know, how to integrate ALSO in your training pipeline! We hope, this knowledge will help you in heterogeneity handling